# NYC IBX parcels (0.3 mile walkshed)

This notebook pulls NYC MapPLUTO parcels that intersect a 0.3 mile walkshed around the
proposed IBX station points. It also normalizes fields into the GeoParquet format used
by CivicMapper (see `docs/parcel-parquet-format.md`).

**Workflow summary**
1. Build a 0.3 mile (1,584 ft) buffer around each station and dissolve.
2. Query the MapPLUTO FeatureServer with a spatial filter and pagination.
3. Normalize fields (assessed land/improvement values, categories).
4. Export GeoParquet in EPSG:4326.


In [8]:
import json
import math
import os
from pathlib import Path

import geopandas as gpd
import pandas as pd
import requests
from shapely.geometry import shape
from shapely.ops import unary_union

# Instead of using a fixed, possibly non-writable directory, use a writable temp directory by default,
# or fall back to a user directory if environment variable is not set.
DATA_DIR = Path(os.environ.get("GEOVIZWIZ_DATA_DIR", "./data"))
try:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
except OSError as e:
    # If the directory cannot be created (read-only file system, etc),
    # fall back to a writable temp directory, or continue (warn).
    import tempfile
    import warnings
    warnings.warn(f"Failed to create DATA_DIR at '{DATA_DIR}': {e}. Using temporary directory instead.")
    DATA_DIR = Path(tempfile.gettempdir()) / "nyc_ibx"
    DATA_DIR.mkdir(parents=True, exist_ok=True)

SERVICE_URL = (
    "https://services5.arcgis.com/GfwWNkhOj9bNBqoJ/arcgis/rest/services/"
    "MAPPLUTO/FeatureServer/0/query"
)

BUFFER_MILES = 0.3
BUFFER_FEET = BUFFER_MILES * 5280
IN_CRS = "EPSG:4326"
NYC_CRS = "EPSG:2263"  # NAD83 / NY Long Island (ftUS)

OUTFILE = DATA_DIR / "nyc-ibx-parcels.parquet"
DO_QUERY = True


In [9]:
ibx_geojson = {
    "type": "FeatureCollection",
    "name": "ibx_stations_rough",
    "features": [
        {
            "type": "Feature",
            "properties": {"name": "Brooklyn Army Terminal", "order": 1, "note": "derived from BAT campus centroid; ok"},
            "geometry": {"type": "Point", "coordinates": [-74.0238925, 40.6458822]},
        },
        {
            "type": "Feature",
            "properties": {"name": "4th Avenue", "order": 2, "note": "_"},
            "geometry": {"type": "Point", "coordinates": [-74.0216953, 40.6377108]},
        },
        {
            "type": "Feature",
            "properties": {"name": "8th Avenue", "order": 3, "note": "near 8 Av (D)"},
            "geometry": {"type": "Point", "coordinates": [-74.0104546, 40.6345048]},
        },
        {
            "type": "Feature",
            "properties": {"name": "New Utrecht Avenue", "order": 4, "note": "62 St / New Utrecht complex"},
            "geometry": {"type": "Point", "coordinates": [-73.9967401, 40.6264583]},
        },
        {
            "type": "Feature",
            "properties": {"name": "McDonald Avenue", "order": 5, "note": "intersection; refine in QGIS if you care"},
            "geometry": {"type": "Point", "coordinates": [-73.9765055, 40.6272726]},
        },
        {
            "type": "Feature",
            "properties": {"name": "East 16th Street", "order": 6, "note": "near Avenue H (Q)"},
            "geometry": {"type": "Point", "coordinates": [-73.9611847, 40.6290802]},
        },
        {
            "type": "Feature",
            "properties": {"name": "Flatbush - Nostrand Avenue", "order": 7, "note": "Flatbush Av–Brooklyn College (2/5)"},
            "geometry": {"type": "Point", "coordinates": [-73.9472158, 40.6305947]},
        },
        {
            "type": "Feature",
            "properties": {"name": "Utica Avenue", "order": 8, "note": "Utica Av (A/C)"},
            "geometry": {"type": "Point", "coordinates": [-73.9288051, 40.6367989]},
        },
        {
            "type": "Feature",
            "properties": {"name": "Remsen Avenue", "order": 9, "note": "ROUGH: snap to Bay Ridge Branch × Remsen in QGIS"},
            "geometry": {"type": "Point", "coordinates": [-73.9136535, 40.6473095]},
        },
        {
            "type": "Feature",
            "properties": {"name": "Linden Boulevard", "order": 10, "note": "New Lots Ave?"},
            "geometry": {"type": "Point", "coordinates": [-73.8996841, 40.6587989]},
        },
        {
            "type": "Feature",
            "properties": {"name": "Livonia Avenue", "order": 11, "note": "Livonia Av (L)"},
            "geometry": {"type": "Point", "coordinates": [-73.9007495, 40.6637123]},
        },
        {
            "type": "Feature",
            "properties": {"name": "Sutter Avenue", "order": 12, "note": "Sutter Av (L)"},
            "geometry": {"type": "Point", "coordinates": [-73.9019833, 40.6685168]},
        },
        {
            "type": "Feature",
            "properties": {"name": "Atlantic Avenue", "order": 13, "note": "Atlantic Av (L) / ENY node"},
            "geometry": {"type": "Point", "coordinates": [-73.9037267, 40.6752057]},
        },
        {
            "type": "Feature",
            "properties": {"name": "Wilson Avenue", "order": 14, "note": "Wilson Av (L)"},
            "geometry": {"type": "Point", "coordinates": [-73.9045413, 40.6886448]},
        },
        {
            "type": "Feature",
            "properties": {"name": "Myrtle Avenue", "order": 15, "note": "near Myrtle–Wyckoff (L/M); schematic"},
            "geometry": {"type": "Point", "coordinates": [-73.8941862, 40.7008689]},
        },
        {
            "type": "Feature",
            "properties": {"name": "Metropolitan Avenue", "order": 16, "note": "Middle Village–Metropolitan Av (M)"},
            "geometry": {"type": "Point", "coordinates": [-73.8888175, 40.7121317]},
        },
        {
            "type": "Feature",
            "properties": {"name": "Eliot Avenue", "order": 17, "note": "ROUGH: snap to ROW × Eliot in QGIS"},
            "geometry": {"type": "Point", "coordinates": [-73.8849122, 40.7224102]},
        },
        {
            "type": "Feature",
            "properties": {"name": "Grand Avenue", "order": 18, "note": "Grand Av–Newtown (M/R)"},
            "geometry": {"type": "Point", "coordinates": [-73.8863499, 40.7305942]},
        },
        {
            "type": "Feature",
            "properties": {"name": "Roosevelt Avenue", "order": 19, "note": "Jackson Heights–Roosevelt Av / 74 St"},
            "geometry": {"type": "Point", "coordinates": [-73.8914289, 40.7467033]},
        },
    ],
}

stations_gdf = gpd.GeoDataFrame.from_features(ibx_geojson["features"], crs=IN_CRS)
stations_gdf.head()


,geometry,name,order,note
0,POINT (-74.02389 40.64588),Brooklyn Army Terminal,1,derived from BAT campus centroid; ok
1,POINT (-74.0217 40.63771),4th Avenue,2,_
2,POINT (-74.01045 40.6345),8th Avenue,3,near 8 Av (D)
3,POINT (-73.99674 40.62646),New Utrecht Avenue,4,62 St / New Utrecht complex
4,POINT (-73.97651 40.62727),McDonald Avenue,5,intersection; refine in QGIS if you care


In [10]:
stations_projected = stations_gdf.to_crs(NYC_CRS)
buffers = stations_projected.buffer(BUFFER_FEET)
walkshed = unary_union(buffers)

walkshed_gdf = gpd.GeoDataFrame(
    [{"geometry": walkshed}], crs=NYC_CRS
)
walkshed_gdf


,geometry
0,"MULTIPOLYGON (((986618.33 166905.876, 986551.8..."


In [11]:
def shapely_to_esri_json(geom):
    """Convert a (Multi)Polygon to ESRI JSON rings."""
    if geom.is_empty:
        raise ValueError("Geometry is empty")

    if geom.geom_type == "Polygon":
        rings = [list(geom.exterior.coords)]
        rings += [list(interior.coords) for interior in geom.interiors]
    elif geom.geom_type == "MultiPolygon":
        rings = []
        for poly in geom.geoms:
            rings.append(list(poly.exterior.coords))
            rings += [list(interior.coords) for interior in poly.interiors]
    else:
        raise ValueError(f"Unsupported geometry type: {geom.geom_type}")

    return {"rings": rings, "spatialReference": {"wkid": 2263}}

geometry_filter = shapely_to_esri_json(walkshed)
geometry_filter


{'rings': [[(986618.330053612, 166905.87590227005),
   (986551.8681568418, 166765.354028004),
   (986471.9527400014, 166632.0232100333),
   (986379.3534322407, 166507.16749702115),
   (986274.9620155136, 166391.98931773688),
   (986159.7838362294, 166287.59790100978),
   (986034.9281232172, 166194.99859324913),
   (985901.5973052465, 166115.08317640857),
   (985761.0754309804, 166048.62127963846),
   (985614.7158028851, 165996.25296733654),
   (985463.9279441877, 165958.48257497765),
   (985310.1640243961, 165935.6738520876),
   (985154.9048741141, 165928.04645913636),
   (984999.6457238321, 165935.6738520876),
   (984845.8818040405, 165958.48257497765),
   (984695.093945343, 165996.25296733654),
   (984548.7343172478, 166048.62127963846),
   (984408.2124429817, 166115.08317640857),
   (984274.881625011, 166194.99859324913),
   (984150.0259119988, 166287.59790100978),
   (984034.8477327146, 166391.98931773685),
   (983930.4563159875, 166507.16749702115),
   (983837.8570082268, 166632.0

In [13]:
OUT_FIELDS = [
    "OBJECTID",
    "BBL",
    "Borough",
    "Block",
    "Lot",
    "Address",
    "OwnerName",
    "LandUse",
    "BldgClass",
    "LotArea",
    "BldgArea",
    "AssessLand",
    "AssessTot",
    "YearBuilt",
    "NumFloors",
    "UnitsRes",
    "UnitsTotal",
]

def query_mappluto(geometry_json, out_fields, batch_size=2000):
    all_features = []
    offset = 0
    while True:
        params = {
            "where": "1=1",
            "geometry": json.dumps(geometry_json),
            "geometryType": "esriGeometryPolygon",
            "spatialRel": "esriSpatialRelIntersects",
            "inSR": 2263,
            "outSR": 4326,
            "outFields": ",".join(out_fields),
            "returnGeometry": "true",
            "f": "geojson",
            "resultOffset": offset,
            "resultRecordCount": batch_size,
            "orderByFields": "OBJECTID",
        }
        # Use POST to avoid URI length limits for large geometries.
        response = requests.post(SERVICE_URL, data=params, timeout=60)
        response.raise_for_status()
        payload = response.json()
        features = payload.get("features", [])
        if not features:
            break
        all_features.extend(features)
        offset += batch_size
        print(f"Fetched {len(features):,} features (offset {offset})")
    return gpd.GeoDataFrame.from_features(all_features, crs=IN_CRS)

if DO_QUERY:
    parcels_gdf = query_mappluto(geometry_filter, OUT_FIELDS)
    print(f"Total parcels fetched: {len(parcels_gdf):,}")
else:
    print("Set SCRAPE_DATA=1 to run the download step.")


Fetched 2,000 features (offset 2000)
Fetched 2,000 features (offset 4000)
Fetched 2,000 features (offset 6000)
Fetched 2,000 features (offset 8000)
Fetched 2,000 features (offset 10000)
Fetched 2,000 features (offset 12000)
Fetched 2,000 features (offset 14000)
Fetched 2,000 features (offset 16000)
Fetched 2,000 features (offset 18000)
Fetched 2,000 features (offset 20000)
Fetched 2,000 features (offset 22000)
Fetched 673 features (offset 24000)
Total parcels fetched: 22,673


In [14]:
def categorize_property_category(land_use, bldg_class):
    if pd.isna(land_use):
        return None
    land_use = str(land_use)
    bldg_class = str(bldg_class) if pd.notna(bldg_class) else ""

    land_use_map = {
        "01": "Single-Family",
        "02": "Two-Family",
        "03": "Three-Family",
        "04": "Multi-Family",
        "05": "Mixed Residential",
        "06": "Commercial",
        "07": "Commercial",
        "08": "Industrial",
        "09": "Parking Lot",
        "10": "Public Facilities",
        "11": "Public Facilities",
    }

    if land_use in land_use_map:
        return land_use_map[land_use]
    if bldg_class.startswith("V"):
        return "Vacant"
    return None

def categorize_property_refined(row):
    cat = str(row.get("PROPERTY_CATEGORY") or "")
    if "Vacant" in cat:
        return "Vacant"
    if "Parking" in cat:
        return "Parking Lot"

    land_value = row.get("REALLANDVA")
    improvement_value = row.get("REALIMPROV")
    if pd.isna(land_value) or pd.isna(improvement_value):
        return None
    if improvement_value < 0.5 * (land_value + improvement_value):
        return "Underdeveloped"
    return None

if DO_QUERY:
    parcels_gdf = parcels_gdf[parcels_gdf.geometry.notnull()].copy()
    parcels_gdf = parcels_gdf.to_crs(IN_CRS)

    parcels_gdf["REALLANDVA"] = parcels_gdf["AssessLand"]
    parcels_gdf["REALIMPROV"] = parcels_gdf["AssessTot"] - parcels_gdf["AssessLand"]

    parcels_gdf["PROPERTY_CATEGORY"] = parcels_gdf.apply(
        lambda row: categorize_property_category(row.get("LandUse"), row.get("BldgClass")),
        axis=1,
    )
    parcels_gdf["property_land_use_refined"] = parcels_gdf.apply(
        categorize_property_refined, axis=1
    )

    parcels_gdf["REALLANDVA_per_sqft"] = parcels_gdf["REALLANDVA"] / parcels_gdf["LotArea"]
    parcels_gdf["REALIMPROV_per_sqft"] = parcels_gdf["REALIMPROV"] / parcels_gdf["LotArea"]
    parcels_gdf["TLLDIMPROV_per_sqft"] = (
        (parcels_gdf["REALLANDVA"] + parcels_gdf["REALIMPROV"]) / parcels_gdf["LotArea"]
    )

    parcels_gdf.to_parquet(OUTFILE, index=False)
    print(f"Saved: {OUTFILE}")
else:
    print("Skipping normalization/export because DO_QUERY is false.")


Saved: data/nyc-ibx-parcels.parquet
